# **Propensity score: matching, weighting and the doubly robust estimate**

Practice for the module [“Causality and Linear Models”](https://ai-interpretability.school), lesson “Propensity score: matching and weighting”.

In the lesson the propensity score methods were formulas and a table of six employees. Here we compute them on the same data and compare them with the true effect. We check four claims of the lesson:

1. matching estimates the effect for the treated group (ATT), weighting for everyone (ATE), and with a heterogeneous effect these are different numbers;
2. the Hájek estimate is more stable than the Horvitz–Thompson one;
3. the balance check detects a wrong propensity score model, while a high ROC-AUC of this model proves nothing;
4. the doubly robust estimate (AIPW) is unbiased if at least one of the two models is correct, and biased if both are wrong.

Plan of the notebook:

1. data with known potential outcomes;
2. the propensity score model;
3. matching (PSM);
4. weighting: Horvitz–Thompson and Hájek;
5. the balance check;
6. AIPW with correct and wrong models;
7. where it breaks: violated positivity, clipping and trimming;
8. the same estimates in DoWhy.

The notebook runs on a CPU, the whole run takes less than a minute.

In [ ]:
%pip install -q dowhy==0.14

In [ ]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. The data

A company sends employees to a professional training course ($T=1$) and a year later looks at the salary raise $Y$ in thousands. Features:

- $X_1$ is tenure (standardized);
- $X_2$ is last year's rating (standardized). Employees with extreme ratings, both very low and very high, are sent to the course more often, and they also get higher raises, so both models contain $Q = X_2^2 - 1$;
- $X_3$ is whether the employee works at the head office.

$$\operatorname{logit} e(x) = -0.3 + 0.7X_1 + 0.4Q + 0.4X_3,$$
$$Y_0 = 20 + 2X_1 + 3Q + 1.5X_3 + \varepsilon,\quad \varepsilon\sim\mathcal N(0, 2^2),\qquad Y_1 = Y_0 + \tau(x),\quad \tau(x) = 2 + X_1.$$

The effect is heterogeneous: the course gives more to experienced employees. Experienced employees are sent to the course more often, so ATT is larger than ATE. Both potential outcomes are generated, and the true ATE and ATT on this sample are computed directly from them.

In [ ]:
def simulate(n=10000, k=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    X1 = rng.normal(size=n); X2 = rng.normal(size=n); X3 = rng.binomial(1, 0.4, n)
    Q = X2 ** 2 - 1
    e = 1 / (1 + np.exp(-k * (-0.3 + 0.7 * X1 + 0.4 * Q + 0.4 * X3)))   # k > 1 makes the groups less alike
    T = rng.binomial(1, e)
    Y0 = 20 + 2 * X1 + 3 * Q + 1.5 * X3 + rng.normal(scale=2, size=n)
    Y1 = Y0 + 2 + X1
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, Y=np.where(T == 1, Y1, Y0), Y0=Y0, Y1=Y1, e_true=e))

d = simulate()
t = d['T'].values == 1
true_ate = (d.Y1 - d.Y0).mean()
true_att = (d.Y1 - d.Y0)[t].mean()
naive = d.Y[t].mean() - d.Y[~t].mean()
print(f'objects {len(d)}, treated {t.sum()} ({t.mean():.3f})')
print(f'true ATE {true_ate:.3f} | true ATT {true_att:.3f} | difference of means {naive:.3f}')

## 2. The propensity score model

Two logistic regressions: a correct one, with the term $X_2^2$, and a wrong one, where $X_2$ enters linearly, as one usually writes it without knowing that extreme ratings matter. The columns `Y0`, `Y1` and `e_true` do not enter the models: real data do not have them.

In [ ]:
PS_FORMULA = {'correct': 'T ~ X1 + X3 + I(X2**2)', 'wrong': 'T ~ X1 + X2 + X3'}

def propensity(data, formula):
    return smf.logit(formula, data=data).fit(disp=0).predict(data).values

e_hat = {name: propensity(d, f) for name, f in PS_FORMULA.items()}
for name, e in e_hat.items():
    print(f'{name:8s} ROC-AUC {roc_auc_score(d["T"], e):.3f} | min e {e.min():.3f} | max e {e.max():.3f}')

## 3. Matching

For every treated employee we look for the control with the nearest logit of $\hat e$, with replacement, and drop pairs farther than the caliper, 0.2 standard deviations of the logit ([Austin, 2011](https://doi.org/10.1002/pst.433)). The estimate is the mean difference of outcomes in the pairs:

$$\widehat{ATT} = \frac{1}{N_1}\sum_{i:\,T_i=1}\big(Y_i - Y_{j(i)}\big).$$

In [ ]:
def psm_att(data, e, caliper_sd=0.2):
    t = data['T'].values == 1
    logit = np.log(e / (1 - e))
    nn = NearestNeighbors(n_neighbors=1).fit(logit[~t].reshape(-1, 1))
    dist, idx = nn.kneighbors(logit[t].reshape(-1, 1))
    keep = dist[:, 0] <= caliper_sd * logit.std()
    y = data['Y'].values
    return np.mean(y[t][keep] - y[~t][idx[keep, 0]]), int((~keep).sum())

att_psm, dropped = psm_att(d, e_hat['correct'])
print(f'PSM: ATT {att_psm:.3f} (true ATT {true_att:.3f}, true ATE {true_ate:.3f})')
print(f'without a pair within the caliper: {dropped} of {t.sum()}')

**Task 1. What is the matching estimate of ATT with the correct propensity score model (rounded to three decimal places)? How many treated employees were left without a pair? Which of the two true effects is the estimate closer to, and why?**

## 4. Weighting

The Horvitz–Thompson estimate divides the weighted sums by $n$, the Hájek estimate by the sums of the weights of its group:

$$\widehat{ATE}_{HT} = \frac1n\sum_i\frac{T_iY_i}{\hat e(x_i)} - \frac1n\sum_i\frac{(1-T_i)Y_i}{1-\hat e(x_i)},\qquad
\widehat{ATE}_{H} = \frac{\sum_i T_iY_i/\hat e(x_i)}{\sum_i T_i/\hat e(x_i)} - \frac{\sum_i (1-T_i)Y_i/(1-\hat e(x_i))}{\sum_i (1-T_i)/(1-\hat e(x_i))}.$$

In [ ]:
def ipw(data, e):
    T, Y = data['T'].values, data['Y'].values
    ht = np.mean(T * Y / e) - np.mean((1 - T) * Y / (1 - e))
    hajek = np.sum(T * Y / e) / np.sum(T / e) - np.sum((1 - T) * Y / (1 - e)) / np.sum((1 - T) / (1 - e))
    return ht, hajek

for name, e in e_hat.items():
    ht, hajek = ipw(d, e)
    print(f'{name:8s} model: Horvitz–Thompson {ht:.3f} | Hajek {hajek:.3f} | true ATE {true_ate:.3f}')
w = np.where(t, 1 / e_hat['correct'], 1 / (1 - e_hat['correct']))
print(f'largest weight {w.max():.1f}, share of the weight sum held by the 1% of objects with the largest weights {np.sort(w)[-100:].sum() / w.sum():.3f}')

**Task 2. What is the Hájek estimate with the correct propensity score model (rounded to three decimal places)? Which of the two estimates, Horvitz–Thompson or Hájek, is closer to the true ATE, and what does the wrong model give?**

## 5. The balance check

The standardized mean difference of a feature between the groups: before weighting a plain one, after weighting with weighted means and variances. The threshold 0.1 is a common rule of thumb ([Austin, 2011](https://pmc.ncbi.nlm.nih.gov/articles/PMC3144483/)). We check not only $X_1, X_2, X_3$ but also $X_2^2$: balance is checked on functions of features that may enter the outcome as well.

In [ ]:
def smd(x, T, w=None):
    w = np.ones_like(x, dtype=float) if w is None else w
    m1, m0 = np.average(x[T == 1], weights=w[T == 1]), np.average(x[T == 0], weights=w[T == 0])
    v1 = np.average((x[T == 1] - m1) ** 2, weights=w[T == 1])
    v0 = np.average((x[T == 0] - m0) ** 2, weights=w[T == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)

T = d['T'].values
features = {'X1': d.X1.values, 'X2': d.X2.values, 'X3': d.X3.values.astype(float), 'X2^2': d.X2.values ** 2}
balance = pd.DataFrame({'before weighting': {k: smd(v, T) for k, v in features.items()}})
for name, e in e_hat.items():
    w = np.where(T == 1, 1 / e, 1 / (1 - e))
    balance[f'after, {name} model'] = {k: smd(v, T, w) for k, v in features.items()}
balance.round(3)

The wrong model balances the features it contains and leaves a difference in $X_2^2$. ROC-AUC does not help here: the correct model's is higher only by a few hundredths, and only the balance table answers whether the groups have been equalized.

## 6. The doubly robust estimate

AIPW ([Robins, Rotnitzky, and Zhao, 1994](https://doi.org/10.1080/01621459.1994.10476818)) takes the prediction of an outcome model and corrects it with weighted residuals:

$$\widehat{ATE}_{AIPW} = \frac1n\sum_i\Big[\hat\mu_1(x_i) - \hat\mu_0(x_i) + \frac{T_i\big(Y_i - \hat\mu_1(x_i)\big)}{\hat e(x_i)} - \frac{(1-T_i)\big(Y_i - \hat\mu_0(x_i)\big)}{1 - \hat e(x_i)}\Big],$$

where $\hat\mu_1$ and $\hat\mu_0$ are outcome models fitted separately on the treated and the control group. The outcome model can also be correct (with the term $X_2^2$) or wrong. Let us go through all four combinations; next to them is the estimate by the outcome model alone, $\frac1n\sum_i(\hat\mu_1(x_i) - \hat\mu_0(x_i))$.

In [ ]:
OUT_FORMULA = {'correct': 'Y ~ X1 + X3 + I(X2**2)', 'wrong': 'Y ~ X1 + X2 + X3'}

def outcome_models(data, formula):
    mu1 = smf.ols(formula, data=data[data['T'] == 1]).fit().predict(data).values
    mu0 = smf.ols(formula, data=data[data['T'] == 0]).fit().predict(data).values
    return mu1, mu0

def aipw(data, e, mu1, mu0):
    T, Y = data['T'].values, data['Y'].values
    return np.mean(mu1 - mu0 + T * (Y - mu1) / e - (1 - T) * (Y - mu0) / (1 - e))

rows = []
for out_name, f in OUT_FORMULA.items():
    mu1, mu0 = outcome_models(d, f)
    for ps_name, e in e_hat.items():
        rows.append(dict(outcome_model=out_name, ps_model=ps_name,
                         outcome_model_only=round(np.mean(mu1 - mu0), 3), AIPW=round(aipw(d, e, mu1, mu0), 3)))
dr = pd.DataFrame(rows)
print(f'true ATE {true_ate:.3f}')
dr

**Task 3. What is the AIPW estimate when the outcome model is wrong and the propensity score model is correct (rounded to three decimal places)? In which of the four combinations is AIPW biased, and what does the outcome model alone give in that combination?**

The models here are fitted on the same objects on which the estimate is computed. With flexible models (forest, boosting) one cannot do this: their overfitting leaks into the estimate, so predictions for every object are taken from models that did not see it, which is cross-fitting, as in DML from the lesson “IV and DML”.

## 7. Where it breaks: positivity

Let us multiply the coefficients of the assignment model by $k=2.5$: the course is now almost determined by the features, and the groups overlap weakly. The true effect stays the same. Two ways from the lesson “Propensity score: matching and weighting” to deal with extreme weights: clipping, which restricts $\hat e$ to the interval $[0.05, 0.95]$ and keeps everyone; trimming, which drops the objects with $\hat e$ outside this interval ([Crump et al., 2009](https://doi.org/10.1093/biomet/asn055)). The second way changes the population for which the effect is estimated, so the true ATE of the remaining objects is shown next to it.

In [ ]:
d_bad = simulate(k=2.5)
T_bad = d_bad['T'].values
e_bad = propensity(d_bad, PS_FORMULA['correct'])
w_bad = np.where(T_bad == 1, 1 / e_bad, 1 / (1 - e_bad))
ess = w_bad.sum() ** 2 / (w_bad ** 2).sum()
mu1_b, mu0_b = outcome_models(d_bad, OUT_FORMULA['correct'])
true_bad = (d_bad.Y1 - d_bad.Y0).mean()
print(f'ROC-AUC {roc_auc_score(T_bad, e_bad):.3f} | min e {e_bad.min():.4f} | max e {e_bad.max():.4f} | largest weight {w_bad.max():.0f} | effective sample size {ess:.0f} of {len(d_bad)}')

e_clip = np.clip(e_bad, 0.05, 0.95)
keep = (e_bad > 0.05) & (e_bad < 0.95)
rows = [
    dict(method='no correction', Hajek=ipw(d_bad, e_bad)[1], AIPW=aipw(d_bad, e_bad, mu1_b, mu0_b), true=true_bad),
    dict(method='clipping [0.05, 0.95]', Hajek=ipw(d_bad, e_clip)[1], AIPW=aipw(d_bad, e_clip, mu1_b, mu0_b), true=true_bad),
    dict(method=f'trimming, {keep.mean():.3f} kept', Hajek=ipw(d_bad[keep], e_bad[keep])[1],
         AIPW=aipw(d_bad[keep], e_bad[keep], mu1_b[keep], mu0_b[keep]), true=(d_bad.Y1 - d_bad.Y0)[keep].mean()),
]
pd.DataFrame(rows).round(3)

The effective sample size $\big(\sum_i w_i\big)^2/\sum_i w_i^2$ shows how many objects with equal weights the weighted sample is worth. Clipping biases the Hájek estimate: objects with extreme $\hat e$ get less weight than they should, and confounding through them comes back. AIPW holds up better, because the weighting in it is only a correction to the outcome model. Trimming answers a different question, about the effect for objects that have someone to be compared with, and its estimate has to be compared with the true effect of the remaining objects.

## 8. The same estimates in DoWhy

DoWhy fits the propensity score by logistic regression on the common causes it is given, so we pass $X_2^2$ as a separate column. Matching in DoWhy is the nearest neighbor with replacement without a caliper.

In [ ]:
from dowhy import CausalModel

data_dw = d[['X1', 'X3', 'T', 'Y']].assign(X2sq=d.X2 ** 2, T=d['T'].astype(bool))
model = CausalModel(data=data_dw, treatment='T', outcome='Y', common_causes=['X1', 'X2sq', 'X3'])
estimand = model.identify_effect(proceed_when_unidentifiable=True)
psm_dw = model.estimate_effect(estimand, method_name='backdoor.propensity_score_matching', target_units='att')
ipw_dw = model.estimate_effect(estimand, method_name='backdoor.propensity_score_weighting', target_units='ate')
print(f'DoWhy PSM, ATT: {psm_dw.value:.3f} | our PSM with a caliper: {att_psm:.3f}')
print(f'DoWhy IPW, ATE: {ipw_dw.value:.3f} | our Hajek: {ipw(d, e_hat["correct"])[1]:.3f}')

## What to take away

- **Matching and weighting estimate different things.** With a heterogeneous effect ATT and ATE diverge, and the PSM and IPW numbers cannot be compared as two estimates of one effect.
- **Hájek instead of Horvitz–Thompson.** Normalizing by the sum of weights damps part of the spread from large weights.
- **A propensity score model is judged by balance, not by ROC-AUC.** A wrong model balances only the features it contains.
- **AIPW is biased only when both models are wrong.** But both models see the same measured features, and AIPW does not save one from a hidden confounder.
- **Extreme weights signal weak overlap.** Clipping biases the estimate, trimming changes the question; this choice goes into the report together with the number of dropped objects.

Sources:
1. [Rosenbaum and Rubin, 1983. The central role of the propensity score in observational studies for causal effects](https://doi.org/10.1093/biomet/70.1.41)
2. [Robins, Rotnitzky, and Zhao, 1994. Estimation of Regression Coefficients When Some Regressors are not Always Observed](https://doi.org/10.1080/01621459.1994.10476818)
3. [Bang and Robins, 2005. Doubly Robust Estimation in Missing Data and Causal Inference Models](https://doi.org/10.1111/j.1541-0420.2005.00377.x)
4. [Austin, 2011. Optimal caliper widths for propensity-score matching when estimating differences in means and differences in proportions in observational studies](https://doi.org/10.1002/pst.433)
5. [Crump, Hotz, Imbens, and Mitnik, 2009. Dealing with limited overlap in estimation of average treatment effects](https://doi.org/10.1093/biomet/asn055)